In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.backtest import run_backtest
from src.research_validation import portfolio_metrics


# 07 Walk-Forward Out-of-Sample Backtest
Run the full daily strategy using frozen formation parameters and next-close synthetic option execution.


In [ ]:
cfg = ResearchConfig()


In [ ]:
train_prices = pd.read_parquet("train_prices.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
eligible_pairs = pd.read_parquet("eligible_pairs.parquet")
cointegrated_pairs = pd.read_parquet("cointegrated_pairs.parquet")
risk_free_rates = pd.read_parquet("risk_free_rates.parquet").iloc[:, 0]
display(pd.Series(cfg.to_dict()))


In [ ]:
result = run_backtest(
    train_prices,
    test_prices,
    eligible_pairs,
    cointegrated_pairs,
    risk_free_rates,
    cfg,
)
summary = portfolio_metrics(result, cfg.initial_capital)
for name, frame in result.items():
    frame.to_parquet(name + ".parquet")
pd.to_pickle(summary, "backtest_summary.pkl")

display(pd.Series(summary, name="Backtest results"))
display(result["trades"].head(10))
result["equity_curve"].equity.plot(figsize=(11, 4), title="Synthetic option portfolio")
plt.ylabel("Model equity")
plt.show()
